# Export Transformation GIFs

Generates animated GIFs for each target shape and one combined overview GIF.  
Output goes to `Datasaurus_Cpp/gifs/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import os
import glob

FRAME_DIR = '../frames'
OUT_DIR = '../gifs'
os.makedirs(OUT_DIR, exist_ok=True)

FPS = 20
DPI = 120

# Skip non-dense sierpinski (too few target points for a good result)
SKIP_TARGETS = {'sierpinski'}
# Display names for titles
DISPLAY_NAMES = {'sierpinski_dense': 'Sierpinski'}

def get_stats(df):
    return {
        'mean_x': df[0].mean(), 'mean_y': df[1].mean(),
        'std_x': df[0].std(), 'std_y': df[1].std(),
        'corr': df[[0,1]].corr().iloc[0, 1]
    }

def select_frames(all_files, max_frames=200):
    """Adaptive frame selection: dense early, sparse late, always include final."""
    non_final = [f for f in all_files if 'final' not in f]
    final = [f for f in all_files if 'final' in f]
    
    if len(non_final) <= max_frames:
        return non_final + final
    
    # First 100 frames detailed, rest subsampled
    action = non_final[:100]
    remaining = non_final[100:]
    step = max(1, len(remaining) // (max_frames - 100))
    polishing = remaining[::step]
    return action + polishing + final

def display_name(target):
    """Get a clean display name for a target."""
    if target in DISPLAY_NAMES:
        return DISPLAY_NAMES[target]
    return target.replace('_', ' ').title()

all_targets = sorted([d for d in os.listdir(FRAME_DIR) if os.path.isdir(os.path.join(FRAME_DIR, d))])
available = [t for t in all_targets if t not in SKIP_TARGETS]
print(f'Found targets: {all_targets}')
print(f'Using targets: {available} (skipping {SKIP_TARGETS})')
print(f'Output directory: {os.path.abspath(OUT_DIR)}')

## Individual GIFs (one per target shape)

In [ ]:
for target in available:
    frame_dir = os.path.join(FRAME_DIR, target)
    all_files = sorted(glob.glob(os.path.join(frame_dir, 'frame_*.csv')))
    if len(all_files) < 5:
        print(f'Skipping {target}: not enough frames')
        continue

    frames = select_frames(all_files)
    dname = display_name(target)
    
    df_base = pd.read_csv(all_files[0], header=None)
    base_stats = get_stats(df_base)
    point_size = 1.5 if len(df_base) > 500 else 8

    # Side-by-side layout: scatter plot (left) + stats panel (right)
    fig, (ax_plot, ax_stats) = plt.subplots(1, 2, figsize=(10, 5),
                                             gridspec_kw={'width_ratios': [1, 0.7]})

    # Left: scatter plot
    scatter = ax_plot.scatter([], [], s=point_size, color='#2c3e50', alpha=0.7, edgecolors='none')
    ax_plot.set_xlim(0, 100)
    ax_plot.set_ylim(0, 100)
    ax_plot.set_aspect('equal')
    ax_plot.set_title(f'Dino \u2192 {dname}', fontsize=13)
    ax_plot.set_xlabel('X')
    ax_plot.set_ylabel('Y')
    ax_plot.grid(True, alpha=0.3)

    # Right: stats panel (no axes, just large text)
    ax_stats.axis('off')
    stats_text = ax_stats.text(0.05, 0.55, '', transform=ax_stats.transAxes,
                               fontsize=16, verticalalignment='center',
                               family='monospace',
                               bbox=dict(boxstyle='round,pad=0.8', facecolor='#f0f0f0',
                                         edgecolor='#cccccc', alpha=0.95))

    fig.tight_layout()

    def make_update(sc, txt, bs):
        def update(frame_path):
            df = pd.read_csv(frame_path, header=None)
            sc.set_offsets(np.c_[df[0], df[1]])
            mx, my = df[0].mean(), df[1].mean()
            sx, sy = df[0].std(), df[1].std()
            corr = df[[0,1]].corr().iloc[0, 1]
            txt.set_text(
                f'X Mean: {mx:>9.6f}\n'
                f'Y Mean: {my:>9.6f}\n'
                f'X SD  : {sx:>9.6f}\n'
                f'Y SD  : {sy:>9.6f}\n'
                f'Corr. : {corr:>9.6f}')
            return sc, txt
        return update

    ani = FuncAnimation(fig, make_update(scatter, stats_text, base_stats),
                        frames=frames, blit=True, interval=1000//FPS)

    out_path = os.path.join(OUT_DIR, f'{target}.gif')
    ani.save(out_path, writer=PillowWriter(fps=FPS), dpi=DPI)
    plt.close(fig)
    
    size_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f'Saved: {out_path} ({len(frames)} frames, {size_mb:.1f} MB)')

## Combined GIF (all targets side by side)

In [ ]:
# Collect all targets that have enough frames
targets_with_data = []
for target in available:
    files = sorted(glob.glob(os.path.join(FRAME_DIR, target, 'frame_*.csv')))
    if len(files) >= 5:
        targets_with_data.append(target)

n = len(targets_with_data)
if n == 0:
    print('No targets with data found.')
else:
    # Compute grid layout
    cols = min(n, 3)
    rows = (n + cols - 1) // cols

    # Select frames (use shortest set to keep all in sync)
    all_frame_lists = {}
    for t in targets_with_data:
        files = sorted(glob.glob(os.path.join(FRAME_DIR, t, 'frame_*.csv')))
        all_frame_lists[t] = select_frames(files, max_frames=150)

    min_len = min(len(v) for v in all_frame_lists.values())
    for t in all_frame_lists:
        all_frame_lists[t] = all_frame_lists[t][:min_len]

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    if n == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    scatters = []
    for i, t in enumerate(targets_with_data):
        ax = axes[i]
        df0 = pd.read_csv(all_frame_lists[t][0], header=None)
        ps = 1 if len(df0) > 500 else 6
        sc = ax.scatter([], [], s=ps, color='#2c3e50', alpha=0.7, edgecolors='none')
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_aspect('equal')
        ax.set_title(display_name(t), fontsize=11)
        ax.grid(True, alpha=0.3)
        scatters.append(sc)

    # Hide unused axes
    for j in range(n, len(axes)):
        axes[j].axis('off')

    fig.suptitle('Datasaurus Transformations', fontsize=16, fontweight='bold')
    plt.tight_layout()

    def update_all(frame_idx):
        artists = []
        for i, t in enumerate(targets_with_data):
            df = pd.read_csv(all_frame_lists[t][frame_idx], header=None)
            scatters[i].set_offsets(np.c_[df[0], df[1]])
            artists.append(scatters[i])
        return artists

    ani = FuncAnimation(fig, update_all, frames=min_len, blit=True, interval=1000//FPS)

    out_path = os.path.join(OUT_DIR, 'all_transformations.gif')
    ani.save(out_path, writer=PillowWriter(fps=FPS), dpi=100)
    plt.close(fig)

    size_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f'Saved: {out_path} ({min_len} frames, {size_mb:.1f} MB)')